In [1]:
import sys
import subprocess

# This forces the install into the CURRENT active kernel
print(">>> Force Installing Selenium to current kernel...")
subprocess.check_call([sys.executable, "-m", "pip", "install", "selenium", "webdriver-manager", "pandas", "lxml", "openpyxl"])
print(">>> Install Complete. Now running imports...")

# Now import
import selenium
import pandas as pd
from selenium import webdriver

print("SUCCESS! Selenium version:", selenium.__version__)
print("You can now run the main script.")

>>> Force Installing Selenium to current kernel...
>>> Install Complete. Now running imports...
SUCCESS! Selenium version: 4.40.0
You can now run the main script.


In [2]:
import sys
import subprocess

# Force install the specific stealth libraries into the current Notebook kernel
libraries = ["undetected-chromedriver", "fake-useragent", "pandas", "lxml", "openpyxl"]

print(f">>> Installing {libraries} to current kernel...")
subprocess.check_call([sys.executable, "-m", "pip", "install"] + libraries)
print(">>> Install Complete. Restarting imports...")

import undetected_chromedriver as uc
from fake_useragent import UserAgent
print(">>> SUCCESS! Stealth libraries are now ready.")

>>> Installing ['undetected-chromedriver', 'fake-useragent', 'pandas', 'lxml', 'openpyxl'] to current kernel...
>>> Install Complete. Restarting imports...
>>> SUCCESS! Stealth libraries are now ready.


# This code scrapped the Agriculture Marketing Information Service of Punjab (AMIS) website. and returned the prices data of agricultural commodities

In [ ]:
import time
import random
import pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# --- CONFIGURATION ---
# The URLs you confirmed
CITY_URLS = {
    "Lahore": "http://www.amis.pk/ViewPrices.aspx?searchType=1&commodityId=1",
    "Faisalabad": "http://www.amis.pk/ViewPrices.aspx?searchType=1&commodityId=2",
    "Rawalpindi": "http://www.amis.pk/ViewPrices.aspx?searchType=1&commodityId=6",
    "Multan": "http://www.amis.pk/ViewPrices.aspx?searchType=1&commodityId=7"
}

# The 5 commodities we need for the Magistrate (Spelling must match site output)
# Note: We filter these *after* we grab the full table
TARGET_ITEMS = ["Potato", "Onion", "Tomato", "Sugar", "Wheat", "Banana"]

def get_date_list():
    # Last 30 days in M/D/YYYY format (matching your screenshot 2/16/2026)
    return [d.strftime('%m/%d/%Y') for d in pd.date_range(end=pd.Timestamp.now(), periods=30)]

if __name__ == "__main__":
    print(">>> INITIALIZING SURGICAL SCRAPER...")
    driver = uc.Chrome()
    all_data = []

    try:
        for city, url in CITY_URLS.items():
            print(f"\n>>> PROCESSING CITY: {city}")
            driver.get(url)
            
            # Initial load wait
            time.sleep(5)

            for date_str in get_date_list():
                try:
                    print(f"  > Target Date: {date_str}", end=" ... ")
                    
                    # 1. FIND DATE BOX (Using the ID)
                    date_input = WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.ID, "ctl00_cphPage_DateTextBox"))
                    )
                    
                    # Clear and Enter Date
                    driver.execute_script("arguments[0].value = '';", date_input)
                    date_input.send_keys(date_str)

                    # 2. CLICK BUTTON (Using ID)
                    
                    search_btn = driver.find_element(By.ID, "ctl00_cphPage_ReminderButton")
                    driver.execute_script("arguments[0].click();", search_btn)
                    
                    # 3. WAIT FOR TABLE 
                    # This confirms the data actually loaded
                    WebDriverWait(driver, 10).until(
                        EC.presence_of_element_located((By.ID, "ctl00_cphPage_Grd"))
                    )
                    
                    # 4. PARSE DATA
                    # We grab the HTML of the table directly
                    table_html = driver.find_element(By.ID, "ctl00_cphPage_Grd").get_attribute('outerHTML')
                    df = pd.read_html(table_html)[0]
                    
                    # 5. CLEAN & FILTER
                    # Add metadata
                    df['City'] = city
                    df['Date'] = date_str
                    
                    # Filter for our specific commodities (Case Insensitive)
                    # This handles "Potato Fresh", "Potato", "Tomato", etc.
                    mask = df.astype(str).apply(lambda x: x.str.contains('|'.join(TARGET_ITEMS), case=False)).any(axis=1)
                    clean_df = df[mask].copy()
                    
                    if not clean_df.empty:
                        all_data.append(clean_df)
                        print(f"SUCCESS ({len(clean_df)} rows)")
                    else:
                        print("EMPTY (No target items found)")

                    # Random "Human" Pause between dates
                    time.sleep(random.uniform(2, 4))

                except Exception as e:
                    print(f"FAILED: {e}")
                    # If it fails, sometimes a refresh helps reset the form state
                    driver.get(url)
                    time.sleep(4)
                    continue

    except Exception as fatal_e:
        print(f"\nCRITICAL ERROR: {fatal_e}")

    finally:
        print("\n>>> CLOSING BROWSER...")
        driver.quit()

        
        if all_data:
            final_df = pd.concat(all_data, ignore_index=True)
            filename = "Punjab_Market_Data_Raw_Final_Banana.xlsx"
            final_df.to_excel(filename, index=False)
            print(f"\n>>> MISSION ACCOMPLISHED.")
            print(f">>> File Saved: {filename}")
            print(f">>> Total Rows: {len(final_df)}")
        else:
            print("\n>>> FAILED. No data collected.")

>>> INITIALIZING SURGICAL SCRAPER...

>>> PROCESSING CITY: Lahore
  > Target Date: 01/23/2026 ... 

C:\Users\hp\AppData\Local\Temp\ipykernel_7416\15593064.py:66: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(table_html)[0]


SUCCESS (13 rows)
  > Target Date: 01/24/2026 ... 

C:\Users\hp\AppData\Local\Temp\ipykernel_7416\15593064.py:66: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(table_html)[0]


SUCCESS (13 rows)
  > Target Date: 01/25/2026 ... 

...

SUCCESS (13 rows)
  > Target Date: 02/19/2026 ... FAILED: Message: stale element reference: stale element not found
  (Session info: chrome=145.0.7632.76); For documentation on this error, please visit: https://www.selenium.dev/documentation/webdriver/troubleshooting/errors#staleelementreferenceexception
Stacktrace:
Symbols not available. Dumping unresolved backtrace:
	0x55f3e3
	0x55f424
	0x352090
	0x363fb2
	0x36307c
	0x359629
	0x35970d
	0x357a42
	0x35b252
	0x3dc851
	0x3bee1c
	0x3dbb23
	0x3bebb6
	0x390319
	0x3910d4
	0x7b46a4
	0x7af9a9
	0x7cc940
	0x5780f8
	0x57fe4d
	0x567c18
	0x567de2
	0x5518ea
	0x76425d49
	0x76eed83b
	0x76eed7c1

  > Target Date: 02/20/2026 ... 

C:\Users\hp\AppData\Local\Temp\ipykernel_7416\15593064.py:66: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(table_html)[0]


SUCCESS (13 rows)
  > Target Date: 02/21/2026 ... 

C:\Users\hp\AppData\Local\Temp\ipykernel_7416\15593064.py:66: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  df = pd.read_html(table_html)[0]


SUCCESS (13 rows)

>>> CLOSING BROWSER...

>>> MISSION ACCOMPLISHED.
>>> File Saved: Punjab_Market_Data_Raw_Final_Banana.xlsx
>>> Total Rows: 1417


# This is a web scraper to get crop arrivals data, but it resulted in all data aggregated in a single column

In [ ]:
import time
import pandas as pd
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException, InvalidSessionIdException, WebDriverException

TARGET_URL = "http://www.amis.pk/Arrivalreports/ArrivalDistrict.aspx"
TARGET_DISTRICTS = ["Lahore", "Rawalpindi", "Faisalabad", "Multan"]

def get_date_list():
    # M/D/YYYY format, 30 periods ending today
    return [d.strftime('%m/%d/%Y').lstrip("0").replace("/0", "/") for d in pd.date_range(end=pd.Timestamp.now(), periods=30)]

def create_driver():
    """Spins up a fresh, resource-optimized Chrome instance."""
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage") # Prevents memory leaks/crashes
    # options.add_argument('--headless') # Uncomment to run silently
    return uc.Chrome(options=options)

if __name__ == "__main__":
    print(">>> INITIALIZING ARRIVALS SCRAPER V3.1 (AJAX-STABLE ENGINE)...")
    driver = create_driver()
    all_arrivals = []

    try:
        driver.get(TARGET_URL)
        WebDriverWait(driver, 15).until(
            EC.presence_of_element_located((By.ID, "allmarket_ctl04_ctl03_txtValue"))
        )

        for date_str in get_date_list():
            max_retries = 3
            
            for attempt in range(max_retries):
                try:
                    attempt_label = f" (Attempt {attempt + 1})" if attempt > 0 else ""
                    print(f">>> Extracting Date: {date_str}{attempt_label} ...", end=" ")
                    
                    # 1. Inject Date
                    date_input = WebDriverWait(driver, 15).until(
                        EC.element_to_be_clickable((By.ID, "allmarket_ctl04_ctl03_txtValue"))
                    )
                    driver.execute_script("arguments[0].value = '';", date_input)
                    date_input.send_keys(date_str)

                    # 2. Grab a reference to the old table (if it exists) to watch it die
                    try:
                        old_table_indicator = driver.find_element(By.XPATH, f"//tr[td[contains(., '{TARGET_DISTRICTS[0]}')]]")
                    except:
                        old_table_indicator = None

                    # 3. Click Report
                    view_btn = driver.find_element(By.ID, "allmarket_ctl04_ctl00")
                    driver.execute_script("arguments[0].click();", view_btn)
                    
                    # 4. Wait for AJAX to finish (Wait for old table to become stale/disappear)
                    if old_table_indicator:
                        try:
                            WebDriverWait(driver, 5).until(EC.staleness_of(old_table_indicator))
                        except TimeoutException:
                            pass # Table might not have changed, which is fine
                            
                    # Hard pause to let the ASP.NET UpdatePanel fully attach the new DOM
                    time.sleep(3) 

                    # 5. Wait for the NEW table to render
                    city_xpath_condition = " or ".join([f"contains(., '{c}')" for c in TARGET_DISTRICTS])
                    wait_xpath = f"//tr[td[{city_xpath_condition}]]"
                    WebDriverWait(driver, 15).until(
                        EC.presence_of_element_located((By.XPATH, wait_xpath))
                    )

                    # 6. Extract Data safely
                    daily_count = 0
                    for city in TARGET_DISTRICTS:
                        try:
                            row = driver.find_element(By.XPATH, f"//tr[td[contains(text(), '{city}')] or td[contains(., '{city}')]]")
                            cols = row.find_elements(By.TAG_NAME, "td")
                            col_texts = [col.text.strip() for col in cols]
                            
                            if len(col_texts) >= 15: 
                                city_data = {
                                    "Date": date_str,
                                    "City": city,
                                    "Potato_Arrival": col_texts[2],
                                    "Tomato_Arrival": col_texts[3],
                                    "Onion_Arrival": col_texts[4],
                                    "Banana": col_texts[5],
                                    "Wheat_Arrival": col_texts[15]
                                }
                                all_arrivals.append(city_data)
                                daily_count += 1
                        except Exception as e:
                            # If it's a stale element, raise it to trigger a retry. 
                            # Otherwise, the city is just missing, which is normal.
                            if "stale element reference" in str(e).lower():
                                raise e 
                    
                    if daily_count > 0:
                        print(f"SUCCESS ({daily_count} cities)")
                    else:
                        print("EMPTY (No matching cities found in data)")
                        
                    break # Success! Break the attempt loop

                except InvalidSessionIdException:
                    # ONLY reboot if the browser actually crashed/disconnected
                    print(f"\n[!] BROWSER CRASH DETECTED. Rebooting...")
                    try:
                        driver.quit()
                    except:
                        pass
                    driver = create_driver()
                    driver.get(TARGET_URL)
                    time.sleep(3)
                    
                except Exception as e:
                    # Catch Timeouts, Stale Elements, etc., and refresh the page to try again
                    print(f"FAILED ({type(e).__name__}). Refreshing...")
                    try:
                        driver.get(TARGET_URL)
                        time.sleep(4)
                    except InvalidSessionIdException:
                        print("\n[!] Session died during refresh. Will reboot on next attempt.")

            else:
                print(f"\n[!] GIVING UP on {date_str} after {max_retries} failed attempts. Moving on...")

    finally:
        try:
            driver.quit()
        except:
            pass
        
        # --- SAVE THE CLEAN ASSET ---
        if all_arrivals:
            df = pd.DataFrame(all_arrivals)
            df.replace('', '0', inplace=True)
            filename = "Punjab_Arrivals_Cleaned.csv"
            df.to_csv(filename, index=False)
            print(f"\n>>> V3.1 COMPLETE. Extracted {len(df)} total rows.")
            print(f">>> Saved as: {filename}")
        else:
            print("\n>>> No data was extracted.")

>>> INITIALIZING ARRIVALS SCRAPER V3.1 (AJAX-STABLE ENGINE)...
>>> Extracting Date: 1/23/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 1/24/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 1/25/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 1/26/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 1/27/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 1/28/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 1/29/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 1/30/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 1/31/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 2/1/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 2/2/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 2/3/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 2/4/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 2/5/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 2/6/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 2/7/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 2/8/2026 ... SUCCESS (4 cities)
>>> Extracting Date: 2/9/20

# This code fixed the corrupted CSV generated by the previous code. All the data was inside the Potato_Arrival column
transforms a messy data-dump back into a professional, analysis-ready dataset.

In [ ]:
import pandas as pd

def resurrect_data():
    print(">>> INITIALIZING DATA RECOVERY...")
    
    # Load the corrupted V2 file
    df = pd.read_csv("Punjab_Arrivals_Cleaned_Banana_V3.csv")
    
    # Since the entire table was trapped in every row, we only need to parse ONE row per Date
    df_unique = df.drop_duplicates(subset=['Date'])
    
    clean_data = []
    target_cities = ["Lahore", "Rawalpindi", "Faisalabad", "Multan"]
    
    for index, row in df_unique.iterrows():
        date_val = row['Date']
        
        # The entire ASP.NET table is trapped inside this single string
        raw_text = str(row['Potato_Arrival'])
        
        # Split the massive string by Newlines (\n)
        lines = [line.strip() for line in raw_text.split('\n') if line.strip() != '']
        
        try:
            # The actual data starts right after the last header ('Paddy')
            start_idx = lines.index('Paddy') + 1
            data_lines = lines[start_idx:]
            
            # The table has exactly 19 data points per row
            chunk_size = 19
            
            # Loop through the massive list in chunks of 19
            for i in range(0, len(data_lines) - chunk_size, chunk_size):
                chunk = data_lines[i:i+chunk_size]
                
                # Verify it is a valid row (The first item must be a Serial Number like '1')
                if len(chunk) == 19 and chunk[0].isdigit():
                    city_name = chunk[1]
                    
                    if city_name in target_cities:
                        # Map the chunk indexes to their true crop values
                        clean_data.append({
                            "Date": date_val,
                            "City": city_name,
                            "Potato_Arrival": chunk[2],
                            "Tomato_Arrival": chunk[3],
                            "Onion_Arrival": chunk[4],
                            "Banana_Arrival": chunk[5],
                            "Wheat_Arrival": chunk[15]
                        })
        except Exception as e:
            # Skip if a day's data is truly corrupted
            pass
            
    final_df = pd.DataFrame(clean_data)
    
    # Save the absolute, final, perfect data
    output_filename = "Punjab_Arrivals_Resurrected.csv"
    final_df.to_csv(output_filename, index=False)
    
    print(f"\n>>> RECOVERY COMPLETE.")
    print(f">>> {len(final_df)} clean rows salvaged perfectly.")
    print(f">>> File Saved: {output_filename}")

if __name__ == "__main__":
    resurrect_data()

>>> INITIALIZING DATA RESURRECTION PROTOCOL...

>>> RESURRECTION COMPLETE.
>>> 104 clean rows salvaged perfectly.
>>> File Saved: Punjab_Arrivals_Resurrected_Banana.csv
